# Spikecurate

_via regression model_

**author**: laquitainesteeve@gmail.com

**purpose**: curate the "good" units (>80% agreement score) amongst the sorted single-units, for the biophysical simulation in the evoked regime, with the two best models so far: a logistic regression (interpretable) and an extra-trees classifier (best overall). Both use the 10 quality metrics plus new features of the unit firing pattern, amplitude distribution, footprint and similarity with other units.

**tested on**:

* Ubuntu 24.04.1 LTS (32 cores, 188 GB RAM, Intel(R) Core(TM) i9-14900K ＠3.2 GHz/5.8 GHz) with RTX 5090 GPU with 40GB VRAM (GPU not needed)

**Method**
- kilosort4's sorting extractor was generated with rtx5090 GPU, so the results slightly change from the original Tesla V100
- a dataset dataframe that contains the sorted single-units (indices), their quality metrics and their quality label ("good" or "bad" units evaluated with our ground truth, columns)

**Execution time**: 3 min

## Setup 

1. Create and activate envs/spikebias.yml

In [1]:
# automatically reload notebook
%load_ext autoreload
%autoreload 2

# import built-in packages
import os

# limit BLAS/OpenMP threads per worker, BEFORE numpy is imported (restart the kernel
# if numpy was already imported): the PCA runs ~20x faster than with default threads
# Each worker runs many small PCA fits, each fit makes many BLAS/LAPACK calls, by 
# default each call can use up to 32 threads. We capped that to 1 thread, which 
# substantially removed overhead and sped up compute.
# 2 min instead of 1.5h
for var in ["OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"]:
    os.environ[var] = "1"
    
import shutil
import pickle
import copy
import random
import pandas as pd
import spikeinterface as si
import spikeinterface.preprocessing as spre
import spikeinterface.core.template_tools as ttools
from spikeinterface import comparison
from spikeinterface.curation import remove_excess_spikes
from spikeinterface import extract_waveforms
from spikeinterface.postprocessing import compute_principal_components
from spikeinterface.qualitymetrics import compute_quality_metrics as qm
import statsmodels.api as sm
from statsmodels.tools.validation import float_like
import torch
import numpy as np
from matplotlib import pyplot as plt
import sklearn
import seaborn as sns
from sklearn import metrics
from sklearn.preprocessing import StandardScaler
import warnings
import spikecurate

# silence irrelevant warnings
warnings.filterwarnings("ignore", message="There is no Probe attached to this recording.*")
warnings.filterwarnings("ignore", message="Versions are not the same.*")

# spikeinterface version
print("Spikeinterface", si.__version__)

# set project path
proj_path = "/home/steeve/steeve/epfl/code/spikebias/"
os.chdir(proj_path)
os.environ["PYTHONPATH"] = proj_path     # worker processes (joblib) must be able to import the project

# import custom package
# (helpers that read sorting_quality.csv are not in the spikecurate package)
from src.nodes.models.Flc import dataloader
from src.nodes.models.Flc import extra_features as ef             # candidate features
from src.nodes.models.Flc import curation_models as cm           # curation models + nested cross-validation

# experiment parameters
EXP = "E"
SORTER = "KS4"

# waveform extraction parameters
FREQ_MIN = 300
REFERENCE = "global"
OPERATOR = "median"
MS_BEFORE = 6
MS_AFTER = 6
LOAD_WE = False
DURATION_SEC = 600          # 10 minutes recording
MAX_SPIKES_PER_UNIT = 500   # max number of spikes used per unit
SEED = 0
N_COMP_PCA = 5              # number of pca components
SPARSE = True
OVERWRITE = True
WE_JOB_KWARGS = dict(n_jobs=20, chunk_duration="1s", progress_bar=True)
job_kwargs = dict(n_jobs=-1, progress_bar=True)
CHUNKS = 80000

# model parameters
model_prms = {"seeds": np.arange(0,100,1), "scale_data": False}

# recording extractor
RECORDING_PATH = "dataset/00_raw/recording_npx_evoked/"

# kilosort sorting extractor
SORTING_KS4_PATH = "dataset/01_intermediate/sorting/npx_evoked/SortingKS4_10m/"

# ground truth extractors
SORTING_TRUE = "dataset/00_raw/ground_truth_npx_evoked"
SORTING_TRUE_10mn = "dataset/00_raw/ground_truth_npx_evoked_10m"

# waveform extractors
WE_SINGLE_U_PATH = "dataset/01_intermediate/waveforms/npx_evoked/SortingKS4_10m_single_units"

# unit classification
QUALITY_PATH = "dataset/01_intermediate/analysis/sorting_quality/sorting_quality.csv"

# figure parameters
plt.rcParams["font.family"] = "Arial"
plt.rcParams["font.size"] = 6
plt.rcParams["lines.linewidth"] = 0.5
plt.rcParams["axes.linewidth"] = 0.5 
plt.rcParams["axes.spines.top"] = False
plt.rcParams["xtick.major.width"] = 0.5
plt.rcParams["xtick.minor.width"] = 0.5
plt.rcParams["ytick.major.width"] = 0.5
plt.rcParams["ytick.minor.width"] = 0.5
plt.rcParams["xtick.major.size"] = 3.5 * 1.1
plt.rcParams["xtick.minor.size"] = 2 * 1.1
plt.rcParams["ytick.major.size"] = 3.5 * 1.1
plt.rcParams["ytick.minor.size"] = 2 * 1.1

# figure legend parameters
legend_cfg = {"frameon": False, "handletextpad": 0.5}
tight_layout_cfg = {"pad": 0.001}
LG_FRAMEON = False

Spikeinterface 0.100.8


## Load datasets

* Execution time: 27 ms

### Recording extractor

* Execution time: 

In [2]:
%%time

# load and preprocess recording
Recording = si.load_extractor(RECORDING_PATH)
Recording.set_property('location', Recording.get_property('location')[:,:2])
Recording = spre.highpass_filter(Recording, freq_min=FREQ_MIN)
Recording = spre.common_reference(Recording, reference=REFERENCE, operator=OPERATOR)
# keep the first 10 min so that the recording duration matches the sorting
# (duration-dependent quality metrics: firing_rate, firing_range, presence_ratio)
Recording = Recording.frame_slice(start_frame=0, end_frame=int(Recording.get_sampling_frequency() * DURATION_SEC))
display(Recording)

FrameSliceRecording: 384 channels - 20.0kHz - 1 segments - 12,000,000 samples 
                     600.00s (10.00 minutes) - float32 dtype - 17.17 GiB

CPU times: user 3.89 ms, sys: 0 ns, total: 3.89 ms
Wall time: 3.87 ms


### Ground truth extractor

* Execution time: 

In [3]:
%%time

# load ground truth
SortingTrue = si.load_extractor(SORTING_TRUE)
SortingTrue = SortingTrue.frame_slice(start_frame=0, end_frame=int(Recording.get_sampling_frequency() * DURATION_SEC))
SortingTrue.save(folder=SORTING_TRUE_10mn, overwrite=True)
display(SortingTrue)

FrameSliceSorting: 1836 units - 1 segments - 20.0kHz

CPU times: user 635 ms, sys: 26.2 ms, total: 661 ms
Wall time: 661 ms


### Sorting extractor

* Execution time: 20 ms

In [4]:
%%time

# load sorting
Sorting = si.load_extractor(SORTING_KS4_PATH)
Sorting = remove_excess_spikes(Sorting, Recording)
Sorting = Sorting.frame_slice(start_frame=0, end_frame=Recording.get_sampling_frequency() * DURATION_SEC)
display(Sorting)

FrameSliceSorting: 3529 units - 1 segments - 20.0kHz

CPU times: user 1.06 ms, sys: 878 µs, total: 1.94 ms
Wall time: 2.02 ms


### Waveform extractor (Sorted units)

* Execution time: 1h47

In [5]:
%%time 

# select sorted single-units amongst single- and multi-units (KSLABEL="good" for kilosort4)
single_unit_ids = Sorting.unit_ids[np.where(Sorting.get_property("KSLabel") == "good")[0]]
Sorting_single_units = Sorting.select_units(unit_ids=single_unit_ids)

# save their waveforms
We_single_u = extract_waveforms(Recording, Sorting_single_units, WE_SINGLE_U_PATH, mode="folder", sparse=SPARSE, ms_before=MS_BEFORE,
                    ms_after=MS_AFTER, max_spikes_per_unit=MAX_SPIKES_PER_UNIT, unit_batch_size=CHUNKS,
                    overwrite=OVERWRITE, seed=SEED, **WE_JOB_KWARGS)


# save waveform PCA data (the extractor is updated)
pca = compute_principal_components(waveform_extractor=We_single_u, n_components=N_COMP_PCA, mode="by_channel_local", **WE_JOB_KWARGS)

extract waveforms shared_memory multi buffer:   0%|          | 0/600 [00:00<?, ?it/s]

extract waveforms memmap multi buffer:   0%|          | 0/600 [00:00<?, ?it/s]

Fitting PCA:   0%|          | 0/408 [00:00<?, ?it/s]

Projecting waveforms:   0%|          | 0/408 [00:00<?, ?it/s]

CPU times: user 29.6 s, sys: 1min 49s, total: 2min 18s
Wall time: 3min 6s


### Single-unit ids and labels (good/gad)

In [6]:
%%time

# label "good" and "bad" sorted single-units to train the classifier
good_unit_ids, bad_unit_ids = dataloader.get_good_and_bad_units(QUALITY_PATH, exp=EXP, sorter=SORTER)

# report
print("\nSingle-unit ids:", single_unit_ids)
print("\nGood unit ids:", good_unit_ids)
print("\nBad unit ids:", bad_unit_ids)


Single-unit ids: ['98' '109' '114' '118' '135' '136' '138' '149' '153' '161' '166' '169'
 '171' '241' '407' '432' '471' '480' '521' '578' '586' '618' '625' '626'
 '638' '653' '691' '739' '793' '887' '923' '939' '971' '1088' '1117'
 '1120' '1124' '1140' '1149' '1170' '1178' '1180' '1181' '1182' '1183'
 '1184' '1187' '1189' '1190' '1191' '1193' '1194' '1195' '1202' '1205'
 '1206' '1207' '1211' '1212' '1213' '1214' '1216' '1217' '1218' '1221'
 '1224' '1227' '1235' '1239' '1242' '1243' '1244' '1246' '1247' '1248'
 '1249' '1250' '1251' '1255' '1256' '1257' '1260' '1261' '1262' '1264'
 '1265' '1266' '1267' '1271' '1272' '1278' '1280' '1281' '1284' '1293'
 '1294' '1295' '1296' '1300' '1304' '1306' '1308' '1309' '1311' '1312'
 '1315' '1318' '1320' '1324' '1325' '1327' '1330' '1331' '1333' '1335'
 '1336' '1337' '1338' '1339' '1340' '1341' '1347' '1348' '1350' '1351'
 '1353' '1355' '1356' '1359' '1366' '1367' '1370' '1371' '1374' '1375'
 '1376' '1377' '1378' '1382' '1383' '1384' '1385' '1386' '

### Unit-tests

In [7]:
%%time

# sanity check units are correct
single_unit_ids_from_file = dataloader.get_sorted_single_unit_ids_from_file(QUALITY_PATH, EXP, SORTER)
assert len(np.intersect1d(single_unit_ids_from_file, single_unit_ids)) == len(single_unit_ids_from_file), "Units are missing or differ"

CPU times: user 4.86 ms, sys: 0 ns, total: 4.86 ms
Wall time: 4.86 ms


In [8]:
Sorting
len(Sorting.unit_ids[np.where(Sorting.get_property("KSLabel") == "good")[0]])
len(single_unit_ids)

408

## Engineer metrics of cell quality

* Execution time: 2 min

In [9]:
%%time

# engineer a dataset of quality metrics (cols) per cell (rows)
data = spikecurate.load_dataset(single_unit_ids, good_unit_ids, bad_unit_ids, Sorting_single_units, SortingTrue, We_single_u, load_qm_if_exists=False, delta_time=1.3, job_kwargs=WE_JOB_KWARGS)

extract amplitudes:   0%|          | 0/600 [00:00<?, ?it/s]

  0%|          | 0/408 [00:00<?, ?it/s]

extract amplitudes:   0%|          | 0/600 [00:00<?, ?it/s]

CPU times: user 40 s, sys: 20.2 s, total: 1min
Wall time: 1min 30s


In [10]:
%%time 

# measure correlations between features (less is better)
data["dataset"].corr()

CPU times: user 183 µs, sys: 0 ns, total: 183 µs
Wall time: 184 µs


,amplitude_cutoff,firing_range,firing_rate,isi_violations_ratio,rp_contamination,rp_violations,sd_ratio,snr,silhouette,mad_ratio,quality_label
amplitude_cutoff,1.000000,-0.202046,-0.205427,-0.057980,-0.038236,-0.042160,0.434142,0.100609,-0.035163,0.373326,-0.206650
firing_range,-0.202046,1.000000,0.587812,0.135004,0.196219,0.315730,-0.029756,0.101577,0.147967,0.082695,0.141726
firing_rate,-0.205427,0.587812,1.000000,0.364549,0.214179,0.268273,-0.135698,0.229419,0.275944,-0.056552,0.353239
isi_violations_ratio,-0.057980,0.135004,0.364549,1.000000,0.724138,0.606383,-0.066802,0.010063,-0.006035,-0.057604,0.184632
rp_contamination,-0.038236,0.196219,0.214179,0.724138,1.000000,0.908150,-0.028520,0.014148,-0.010419,-0.015045,0.088203
rp_violations,-0.042160,0.315730,0.268273,0.606383,0.908150,1.000000,-0.019374,0.014426,-0.015897,0.002773,0.055228
sd_ratio,0.434142,-0.029756,-0.135698,-0.066802,-0.028520,-0.019374,1.000000,0.225191,0.026409,0.915961,-0.254872
snr,0.100609,0.101577,0.229419,0.010063,0.014148,0.014426,0.225191,1.000000,0.586809,0.095481,0.496493
silhouette,-0.035163,0.147967,0.275944,-0.006035,-0.010419,-0.015897,0.026409,0.586809,1.000000,0.000130,0.569014
mad_ratio,0.373326,0.082695,-0.056552,-0.057604,-0.015045,0.002773,0.915961,0.095481,0.000130,1.000000,-0.310388


## Model: train/detect >80% accurate units

Two models, evaluated with nested cross-validation (10 x repeated stratified 4-fold, 75% train / 25% test). The decision threshold of each model is chosen on the training units only: the lowest threshold whose out-of-fold precision reaches 0.92 (the earlier model's precision).

* **logistic regression**: 10 quality metrics + firing pattern (ISI CV and local variation, Fano factor, empty fraction and trend of 5 s bins, short ISI fraction)
* **extra trees** (best overall): 10 quality metrics + all new features (amplitude distribution, firing pattern, spatial footprint and depth, waveform consistency, similarity with other units, Kilosort cluster properties; see src/nodes/models/Flc/extra_features.py)

In [11]:
%%time

# new features of each sorted single-unit (no ground truth is used)
extra = ef.compute_extra_features(We_single_u, duration=DURATION_SEC)

# dataset: 10 quality metrics + new features (units with a missing feature are dropped; silhouette is NaN for a few units)
dataset = data["dataset"].join(extra).dropna()
print("units:", len(dataset), "of", len(data["dataset"]), "| good:", int(dataset["quality_label"].sum()))

units: 407 of 408 | good: 105
CPU times: user 4.13 s, sys: 160 ms, total: 4.29 s
Wall time: 4.29 s


In [12]:
%%time

# the worker processes must be able to import the project
os.environ["PYTHONPATH"] = os.getcwd()

# train & evaluate the two models w/ nested cross-validation
results = cm.nested_cv(dataset, data["predictors"])
display(cm.summarize(results))

BrokenProcessPool: A task has failed to un-serialize. Please ensure that the arguments of the function are all picklable.

## Source data

* precision and recall of each model on the test units of each cross-validation fold (fig5f)
* saved as .xlsx with the .svg figure in figures/8_source_data/fig5f/

In [13]:
# source data: precision and recall of each model in each fold
df = results[["fold", "model", "precision", "recall"]].copy()
display(df.groupby("model", sort=False).agg(["median", "count"]))

NameError: name 'results' is not defined

In [14]:
def plot_source_data(df: pd.DataFrame):
    """Precision and recall of each model across folds (black dots); median and 95% CI in orange"""

    models = list(dict.fromkeys(df["model"]))
    fig, ax = plt.subplots(1, len(models), figsize=(1.1 * len(models) + 0.2, 1.1), sharey=True)
    for a, model in zip(ax, models):
        d = df[df["model"] == model]
        sns.stripplot(ax=a, data=d[["precision", "recall"]], jitter=0.04, color="k", size=3)
        for x, col in enumerate(["precision", "recall"]):
            a.errorbar(x=x, y=np.nanmedian(d[col]), yerr=1.96 * np.std(d[col]) / np.sqrt(len(d[col])), marker="o",
                       color="orange", markeredgecolor="w", markersize=5, zorder=np.inf)
        a.set_title(model, fontsize=6)

        # disconnect axes (R style)
        a.spines[["right", "top"]].set_visible(False)
        a.spines["bottom"].set_position(("axes", -0.05))
        a.yaxis.set_ticks_position("left")
        a.spines["left"].set_position(("axes", -0.05))
        a.set_ylim([0, 1])
    for a in ax[1:]:
        a.spines["left"].set_visible(False); a.set_yticks([])
    plt.subplots_adjust(wspace=0.2)
    return fig, ax

In [15]:
%%time

# save config
savefig_cfg = {"transparent": True, "dpi": 300}

# plot
fig, ax = plot_source_data(df)

# save figure
os.makedirs("figures/8_source_data/fig5f", exist_ok=True)
plt.savefig("figures/8_source_data/fig5f/fig5f.svg", bbox_inches="tight", **savefig_cfg)

# save source data
df.to_excel("figures/8_source_data/fig5f/fig5f.xlsx", index=False)

NameError: name 'df' is not defined

In [16]:
%%time

# check: replot from the saved source data
df = pd.read_excel("figures/8_source_data/fig5f/fig5f.xlsx")
fig, ax = plot_source_data(df)

KeyError: 'model'